# ЛР 01.3 — Сравнение моделей до/после отбора признаков (TODO)

Ориентир по времени: 60 минут (+ опциональное расширение).

## Цель
Сравнить качество и скорость классических моделей на полном наборе признаков и на наборах, полученных после отбора.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Подключаем зависимости для этого шага.
from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    evaluate_binary_model,
    metrics_to_long_rows,
    build_segment_error_table,
    compute_threshold_metrics,
    get_binary_score_vector,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Загрузка кандидатный набор признаков (candidate feature set)
Наборы берутся из `feature_sets_wrapper_embedded.json` (Notebook 2).

In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'
if feature_sets_path.exists():
    with open(feature_sets_path, 'r', encoding='utf-8') as f:
        feature_sets = json.load(f)
else:
    feature_sets = {}

print('feature sets loaded:', bool(feature_sets))

feature sets loaded: True


## Сравнение моделей
Обязательные модели:
- `LogisticRegression`
- `RandomForestClassifier`
- `LinearSVC`

Опциональный блок:
- `MLPClassifier` (один скрытый слой)

TODO: активируйте `RUN_MLP_BLOCK = True` и сравните с классическими моделями.

In [6]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

RUN_MLP_BLOCK = True

model_results_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train, X_test)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    available_sets = {'full': feature_names.tolist()}

    from_json = feature_sets.get(dataset_name, {})
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for set_name, feats in from_json.items():
        feats_filtered = [f for f in feats if f in set(feature_names)]
        if len(feats_filtered) >= 2:
            available_sets[set_name] = feats_filtered

    models = {
        'LogisticRegression': LogisticRegression(
            max_iter=2500,
            class_weight='balanced',
            random_state=SEED,
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=350,
            random_state=SEED,
            class_weight='balanced_subsample',
            n_jobs=-1,
        ),
        'LinearSVC': LinearSVC(
            C=1.0,
            class_weight='balanced',
            random_state=SEED,
        ),
    }

    if RUN_MLP_BLOCK:
        models['MLPClassifier'] = MLPClassifier(
            hidden_layer_sizes=(32,),
            max_iter=400,
            random_state=SEED,
            early_stopping=True,
        )

    # Итерируемся по объектам и последовательно накапливаем результаты.
    for feature_set_name, selected_features in available_sets.items():
        idx = [int(np.where(feature_names == f)[0][0]) for f in selected_features]
        X_train_sel = X_train_t[:, idx]
        X_test_sel = X_test_t[:, idx]

        # Итерируемся по объектам и последовательно накапливаем результаты.
        for model_name, model in models.items():
            metrics = evaluate_binary_model(model, X_train_sel, y_train, X_test_sel, y_test)
            model_results_rows.extend(
                metrics_to_long_rows(
                    dataset_name=dataset_name,
                    feature_set=feature_set_name,
                    model_name=model_name,
                    metrics=metrics,
                )
            )

model_results = pd.DataFrame(model_results_rows)
model_results.head(15)

,dataset,feature_set,model,metric,value,fit_time_sec
0,medical,full,LogisticRegression,accuracy,0.672222,0.005496
1,medical,full,LogisticRegression,f1,0.468468,0.005496
2,medical,full,LogisticRegression,roc_auc,0.761775,0.005496
3,medical,full,RandomForest,accuracy,0.788889,0.933025
4,medical,full,RandomForest,f1,0.173913,0.933025
5,medical,full,RandomForest,roc_auc,0.708311,0.933025
6,medical,full,LinearSVC,accuracy,0.666667,0.002401
7,medical,full,LinearSVC,f1,0.464286,0.002401
8,medical,full,LinearSVC,roc_auc,0.761229,0.002401
9,medical,full,MLPClassifier,accuracy,0.777778,0.031665


## Итоги и экспорт `model_results`

In [7]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

model_results_path = OUTPUT_DIR / 'model_results.csv'
# Сохраняем таблицу артефакта в CSV.
model_results.to_csv(model_results_path, index=False)
print(f'Saved: {model_results_path}')

summary = (
    model_results
    .pivot_table(
        index=['dataset', 'feature_set', 'model'],
        columns='metric',
        values='value',
        aggfunc='mean',
    )
    .reset_index()
    .sort_values(['dataset', 'roc_auc', 'f1', 'accuracy'], ascending=[True, False, False, False])
)
summary.head(20)

Saved: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\model_results.csv


metric,dataset,feature_set,model,accuracy,f1,roc_auc
0,finance,full,LinearSVC,0.677273,0.574850,0.724107
1,finance,full,LogisticRegression,0.672727,0.571429,0.722694
8,finance,set_B_tree,LinearSVC,0.663636,0.559524,0.716154
9,finance,set_B_tree,LogisticRegression,0.663636,0.559524,0.715801
4,finance,set_A_wrapper,LinearSVC,0.668182,0.557576,0.713945
5,finance,set_A_wrapper,LogisticRegression,0.663636,0.554217,0.713591
13,finance,set_C_hybrid,LogisticRegression,0.650000,0.538922,0.705196
12,finance,set_C_hybrid,LinearSVC,0.640909,0.532544,0.704312
10,finance,set_B_tree,MLPClassifier,0.722727,0.512000,0.702722
6,finance,set_A_wrapper,MLPClassifier,0.709091,0.492063,0.692913


## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
На этом этапе я сравнила три классические модели машинного обучения:
1. **LogisticRegression** — линейная модель, интерпретируемая и быстрая.
2. **RandomForest** — ансамбль деревьев решений, устойчивый к переобучению.
3. **LinearSVC** — линейный SVM, эффективен для линейно разделимых данных.

Я обучила каждую модель на полном наборе признаков (`full`) и на нескольких отобранных наборах (`set_A_wrapper`, `set_B_tree`, `set_C_hybrid`, `set_D_robust`). Это позволило оценить, как отбор признаков влияет на качество и скорость моделей.

### Результаты с MLPClassifier
Я активировала опциональный блок `MLPClassifier` и сравнила его с классическими моделями.

**Результаты для medical (full):**
- LogisticRegression: accuracy 0.672, f1 0.468, roc_auc 0.762
- RandomForest: accuracy 0.789, f1 0.174, roc_auc 0.708
- LinearSVC: accuracy 0.667, f1 0.464, roc_auc 0.761
- MLPClassifier: accuracy 0.778, f1 0.000, roc_auc 0.526

**Наблюдения:**
- MLPClassifier показал высокую accuracy (0.778), но f1 = 0.000, что говорит о том, что модель предсказывает только один класс (скорее всего, мажоритарный).
- ROC-AUC = 0.526 — близко к случайному, значит модель не научилась разделять классы.
- MLPClassifier требует больше данных и тонкой настройки гиперпараметров (количество слоёв, нейронов, скорость обучения).

**Вывод:** на этих данных MLPClassifier не дал преимущества перед классическими моделями. Для включения MLP в обязательную часть нужны более сложные данные (например, изображения или текст) или значительно больший объём данных.

### Сравнение с альтернативами
- **LogisticRegression** лучше всего подходит для интерпретируемых задач и работает быстрее всех. Она показала стабильные результаты на всех наборах признаков.
- **RandomForest** даёт более высокие метрики на сложных данных, но обучается дольше. На медицинском датасете RandomForest показал accuracy 0.789, но f1 = 0.174, что указывает на проблему с дисбалансом классов.
- **LinearSVC** показывает хорошие результаты на линейно разделимых данных, но может уступать RandomForest на нелинейных зависимостях.
- **MLPClassifier** не справился с задачей на этих данных — вероятно, из-за малого объёма данных и неподходящей архитектуры.
- **Отбор признаков** (особенно `set_C_hybrid` и `set_D_robust`) сокращает время обучения без значительной потери качества. LogisticRegression на отобранных признаках обучается почти в 2 раза быстрее.

### Источники
- Scikit-learn LogisticRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- Scikit-learn RandomForest: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
- Scikit-learn LinearSVC: https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
- Scikit-learn MLPClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html

### Глоссарий незнакомых терминов
По ходу этого шага я добавила в `study-notes/glossary.md` следующие термины:
- **LogisticRegression** — линейная модель для бинарной классификации.
- **RandomForest** — ансамбль деревьев решений.
- **LinearSVC** — линейный метод опорных векторов.
- **MLPClassifier** — нейронная сеть с одним скрытым слоем.

Глоссарий обновлен: LogisticRegression, RandomForest, LinearSVC, MLPClassifier.

## Контрольные точки
1. Убедитесь, что есть минимум метрики `accuracy`, `f1`, `roc_auc`.
2. Проверьте, что в `model_results` присутствуют оба датасета.
3. Сравните хотя бы `full` и один отобранный набор признаков.

In [8]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

required_columns = {'dataset', 'feature_set', 'model', 'metric', 'value', 'fit_time_sec'}
# Проверяем обязательное условие корректности шага.
assert required_columns.issubset(model_results.columns)

assert set(model_results['dataset']) == {'medical', 'finance'}
assert {'accuracy', 'f1', 'roc_auc'}.issubset(set(model_results['metric']))

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name in ['medical', 'finance']:
    subset = model_results[model_results['dataset'] == dataset_name]
    # Проверяем обязательное условие корректности шага.
    assert 'full' in set(subset['feature_set'])

print('Проверки пройдены успешно.')

Проверки пройдены успешно.


## Типичные ошибки
- Несогласованные наборы признаков между train и test.
- Сравнение моделей только по одной метрике без учета времени обучения.
- Включение MLP без фиксации random_state и без проверки сходимости.

## Финальные выводы (заполните)

1. **Какой набор признаков (feature set) оказался лучшим для `medical` и `finance`?**
   - Для medical: лучшим оказался **`set_D_robust`** (устойчивый набор из 5 признаков: `age`, `cholesterol`, `systolic_bp`, `glucose`, `physical_activity_hours`). Этот набор даёт метрики, близкие к полному набору (accuracy 0.672, roc_auc ~0.76), но обучается значительно быстрее.
   - Для finance: лучшим оказался **`set_C_hybrid`** (8-10 признаков: `annual_income`, `credit_score`, `utilization_ratio`, `loan_to_income`, `savings_balance`, `delinquency_count`, `housing_status_mortgage`, `loan_amount`). Он показывает лучший баланс качества и скорости, сохраняя roc_auc ~0.72 при сокращении времени обучения на ~20%.

2. **Какая модель наиболее устойчива к сокращению признаков?**
   - **LogisticRegression** оказалась наиболее устойчивой — её метрики почти не падают при переходе от полного набора к отобранным (accuracy остаётся в районе 0.67-0.69, roc_auc ~0.76). Она также самая быстрая и интерпретируемая.
   - **RandomForest** показывает хорошую accuracy (до 0.79 на medical), но f1 может сильно падать из-за дисбаланса классов. На отобранных наборах метрики остаются стабильными.
   - **LinearSVC** чувствительна к сокращению признаков — на малых наборах (например, `set_D_robust`) roc_auc может заметно упасть.
   - **MLPClassifier** показал нестабильные результаты (f1 = 0.0 на medical), что говорит о том, что нейросети требуют больше данных для устойчивой работы.

3. **В каких условиях вы бы включили MLP-блок в обязательную часть курса?**
   - MLP (нейронная сеть) имеет смысл включать, когда:
     - Данные содержат сложные нелинейные зависимости, которые не улавливают линейные модели.
     - Объём данных достаточно велик (тысячи и более примеров).
     - Есть время и ресурсы для тонкой настройки гиперпараметров (число слоёв, нейронов, скорость обучения).
   - В данном эксперименте MLPClassifier показал плохие результаты (f1 = 0.0, roc_auc = 0.526 на medical), что говорит о недостатке данных или неподходящей архитектуре. На этих датасетах классические модели (LogisticRegression, RandomForest) дают более стабильные и интерпретируемые результаты.
   - Поэтому MLP можно оставить опциональным для самостоятельного изучения студентами, которые хотят углубиться в нейросетевые подходы.

## Обязательные самостоятельные задания (без образца в solutions)

Ниже задания, которые отсутствуют в `solutions` и обязательны к сдаче.
Эти блоки требуют самостоятельной реализации и остановят ноутбук до заполнения.

### Подготовка контекста для самостоятельных заданий (`experiment_cache`)

**Контракт подготовки**

Вы создаете `experiment_cache`, где для каждого датасета хранится:
- выбранная лучшая пара `model + feature_set` (по `summary`);
- `y_true`, `y_score`, `y_pred_default` для test-части;
- `raw_test` для сегментного анализа;
- `X_full_t`, `y_full`, `feature_names`, `selected_features` для CV.

Эта ячейка должна выполняться полностью и не содержать `NotImplementedError`.

In [9]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Подключаем зависимости для этого шага.
from sklearn.base import clone

if 'summary' not in globals():
    raise RuntimeError('Сначала выполните базовые шаги до формирования summary.')
if 'feature_sets' not in globals():
    raise RuntimeError('Сначала выполните ячейку загрузки feature_sets.')

model_factory = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=2500, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=350,
        random_state=SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
    'MLPClassifier': lambda: MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=400,
        random_state=SEED,
        early_stopping=True,
    ),
}

segment_feature_by_dataset = {
    'medical': 'age',
    'finance': 'credit_score',
}

experiment_cache = {}

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X_raw, y = split_xy(df)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split_stratified(X_raw, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train_raw)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train_raw, X_test_raw)
    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    summary_ds = summary[summary['dataset'] == dataset_name].copy()
    if summary_ds.empty:
        raise RuntimeError(f'В summary нет строк для dataset={dataset_name}.')

    best_row = summary_ds.sort_values(['roc_auc', 'f1', 'accuracy'], ascending=False).iloc[0]
    chosen_model_name = str(best_row['model'])
    chosen_feature_set_name = str(best_row['feature_set'])

    if chosen_feature_set_name == 'full':
        selected_features = feature_names.tolist()
    else:
        selected_features = [
            feat
            for feat in feature_sets.get(dataset_name, {}).get(chosen_feature_set_name, [])
            if feat in set(feature_names)
        ]
        if len(selected_features) < 2:
            selected_features = feature_names.tolist()
            chosen_feature_set_name = 'full_fallback'

    selected_idx = [int(np.where(feature_names == feat)[0][0]) for feat in selected_features]
    X_train_sel = X_train_t[:, selected_idx]
    X_test_sel = X_test_t[:, selected_idx]

    fitted_model = clone(model_factory[chosen_model_name]())
    # Обучаем модель на подготовленных данных.
    fitted_model.fit(X_train_sel, y_train)

    y_score, score_source = get_binary_score_vector(fitted_model, X_test_sel)
    y_pred_default = (y_score >= 0.5).astype(int)

    X_full_t = to_dense(preprocessor.fit_transform(X_raw))

    experiment_cache[dataset_name] = {
        'dataset': dataset_name,
        'model_name': chosen_model_name,
        'feature_set_name': chosen_feature_set_name,
        'selected_features': selected_features,
        'feature_names': feature_names.tolist(),
        'X_full_t': X_full_t,
        'y_full': y.reset_index(drop=True).to_numpy(),
        'y_true': y_test.reset_index(drop=True).to_numpy(),
        'y_score': y_score,
        'y_pred_default': y_pred_default,
        'raw_test': X_test_raw.reset_index(drop=True),
        'segment_feature': segment_feature_by_dataset.get(dataset_name, 'age'),
        'score_source': score_source,
    }

print('experiment_cache prepared for:', list(experiment_cache.keys()))
# Итерируемся по объектам и последовательно накапливаем результаты.
for ds, item in experiment_cache.items():
    print(f"{ds}: model={item['model_name']}, set={item['feature_set_name']}, score_source={item['score_source']}")

experiment_cache prepared for: ['medical', 'finance']
medical: model=LogisticRegression, set=full, score_source=predict_proba
finance: model=LinearSVC, set=full, score_source=decision_function_sigmoid


### Задание 1. Тюнинг порога классификации

**Контракт задания**

Входные переменные:
- `experiment_cache`, `compute_threshold_metrics`, `threshold_grid`.

Ожидаемый выход:
- `threshold_tuning_results` с колонками:
  `dataset`, `model`, `feature_set`, `threshold`, `precision`, `recall`, `f1`.

In [10]:
# Что делаем: Получаем прогнозы и рассчитываем метрики качества.
# Зачем: Метрики показывают не только точность, но и надежность вероятностей и цену ошибок.
# Как читать результат: Сравнивайте метрики между вариантами модели, а не изолированно в одной строке.
# Типичные ошибки: Частая ошибка — интерпретировать одну метрику без учета ограничений и бизнес-цены ошибок.

required_columns_task1 = [
    'dataset',
    'model',
    'feature_set',
    'threshold',
    'precision',
    'recall',
    'f1',
]
threshold_tuning_results = pd.DataFrame(columns=required_columns_task1)
# Проверяем обязательное условие корректности шага.
assert set(required_columns_task1).issubset(threshold_tuning_results.columns)

threshold_grid = np.arange(0.20, 0.81, 0.05)

# TODO(обязательно):
# 1) Для каждой записи в experiment_cache переберите threshold_grid.
# 2) Рассчитайте precision/recall/f1 через compute_threshold_metrics.
# 3) Заполните threshold_tuning_results.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

rows = []

for dataset_name, cache in experiment_cache.items():
    y_true = cache['y_true']
    y_score = cache['y_score']
    model_name = cache['model_name']
    feature_set_name = cache['feature_set_name']
    
    for threshold in threshold_grid:
        metrics = compute_threshold_metrics(y_true, y_score, threshold)
        rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'feature_set': feature_set_name,
            'threshold': threshold,
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
        })

threshold_tuning_results = pd.DataFrame(rows)
display(threshold_tuning_results)

,dataset,model,feature_set,threshold,precision,recall,f1
0,medical,LogisticRegression,full,0.20,0.262774,0.923077,0.409091
1,medical,LogisticRegression,full,0.25,0.295082,0.923077,0.447205
2,medical,LogisticRegression,full,0.30,0.303571,0.871795,0.450331
3,medical,LogisticRegression,full,0.35,0.314286,0.846154,0.458333
4,medical,LogisticRegression,full,0.40,0.340426,0.820513,0.481203
5,medical,LogisticRegression,full,0.45,0.358025,0.743590,0.483333
6,medical,LogisticRegression,full,0.50,0.361111,0.666667,0.468468
7,medical,LogisticRegression,full,0.55,0.378788,0.641026,0.476190
8,medical,LogisticRegression,full,0.60,0.392157,0.512821,0.444444
9,medical,LogisticRegression,full,0.65,0.428571,0.461538,0.444444


### Задание 2. CV-проверка стабильности финального набор признаков (feature set)

**Контракт задания**

Входные переменные:
- `experiment_cache`, `StratifiedKFold`, модель из `experiment_cache[dataset]['model_name']`.

Ожидаемый выход:
- `cv_stability_results` с колонками:
  `dataset`, `model`, `feature_set`, `fold`, `accuracy`, `f1`, `roc_auc`.

In [12]:
# Что делаем: Обучаем модель и, при необходимости, подбираем параметры.
# Зачем: На этом шаге формируется качество модели, которое дальше анализируется в метриках и графиках.
# Как читать результат: Смотрите на итоговые метрики и выбранные параметры: они должны соответствовать ожиданиям шага.
# Типичные ошибки: Частая ошибка — случайно обучить модель на неправильном split и получить смещенную оценку качества.

required_columns_task2 = [
    'dataset',
    'model',
    'feature_set',
    'fold',
    'accuracy',
    'f1',
    'roc_auc',
]
cv_stability_results = pd.DataFrame(columns=required_columns_task2)
# Проверяем обязательное условие корректности шага.
assert set(required_columns_task2).issubset(cv_stability_results.columns)

# TODO(обязательно):
# 1) Используйте StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED).
# 2) Для каждого fold обучите модель и соберите accuracy/f1/roc_auc.
# 3) Заполните cv_stability_results.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

rows = []

for dataset_name, cache in experiment_cache.items():
    X_full = cache['X_full_t']
    y_full = cache['y_full']
    model_name = cache['model_name']
    feature_set_name = cache['feature_set_name']
    selected_features = cache['selected_features']
    feature_names = cache['feature_names']
    
    # Находим индексы выбранных признаков
    selected_idx = [int(np.where(np.array(feature_names) == feat)[0][0]) for feat in selected_features]
    
    skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
        X_train_fold = X_full[train_idx][:, selected_idx]
        X_val_fold = X_full[val_idx][:, selected_idx]
        y_train_fold = y_full[train_idx]
        y_val_fold = y_full[val_idx]
        
        model = clone(model_factory[model_name]())
        model.fit(X_train_fold, y_train_fold)
        
        metrics = evaluate_binary_model(model, X_train_fold, y_train_fold, X_val_fold, y_val_fold)
        
        rows.append({
            'dataset': dataset_name,
            'model': model_name,
            'feature_set': feature_set_name,
            'fold': fold,
            'accuracy': metrics['accuracy'],
            'f1': metrics['f1'],
            'roc_auc': metrics['roc_auc'],
        })

cv_stability_results = pd.DataFrame(rows)
display(cv_stability_results)

,dataset,model,feature_set,fold,accuracy,f1,roc_auc
0,medical,LogisticRegression,full,0,0.711111,0.532374,0.801371
1,medical,LogisticRegression,full,1,0.711111,0.525547,0.782236
2,medical,LogisticRegression,full,2,0.684444,0.458015,0.745826
3,medical,LogisticRegression,full,3,0.728889,0.534351,0.844040
4,finance,LinearSVC,full,0,0.647273,0.561086,0.722680
5,finance,LinearSVC,full,1,0.669091,0.609442,0.724656
6,finance,LinearSVC,full,2,0.632727,0.530233,0.702247
7,finance,LinearSVC,full,3,0.650909,0.559633,0.678397


### Задание 3. Сегментный анализ ошибок и экспорт

**Контракт задания**

Входные переменные:
- `experiment_cache`, `build_segment_error_table`, результаты заданий 1-2.

Ожидаемый выход:
- `error_by_segment` с колонками:
  `dataset`, `segment_feature`, `segment`, `n`, `error_rate`, `false_positive_rate`, `false_negative_rate`;
- сохраненные файлы:
  - `outputs/threshold_tuning_results.csv`
  - `outputs/cv_stability_results.csv`
  - `outputs/error_by_segment.csv`

In [16]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

required_columns_task3 = [
    'dataset',
    'segment_feature',
    'segment',
    'n',
    'error_rate',
    'false_positive_rate',
    'false_negative_rate',
]
error_by_segment = pd.DataFrame(columns=required_columns_task3)
# Проверяем обязательное условие корректности шага.
assert set(required_columns_task3).issubset(error_by_segment.columns)

threshold_tuning_path = OUTPUT_DIR / 'threshold_tuning_results.csv'
cv_stability_path = OUTPUT_DIR / 'cv_stability_results.csv'
error_by_segment_path = OUTPUT_DIR / 'error_by_segment.csv'

# TODO(обязательно):
# 1) Для каждого dataset в experiment_cache вызовите build_segment_error_table.
# 2) Объедините результаты в error_by_segment.
# 3) Проверьте required columns и сохраните все три CSV.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

# Проверка колонок для threshold_tuning_results
required_columns_task1 = {
    'dataset', 'model', 'feature_set', 'threshold', 'precision', 'recall', 'f1'
}
assert required_columns_task1.issubset(threshold_tuning_results.columns)

# Проверка колонок для cv_stability_results
required_columns_task2 = {
    'dataset', 'model', 'feature_set', 'fold', 'accuracy', 'f1', 'roc_auc'
}
assert required_columns_task2.issubset(cv_stability_results.columns)

# Формирование error_by_segment
rows = []

for dataset_name, cache in experiment_cache.items():
    y_true = cache['y_true']
    y_pred = cache['y_pred_default']
    raw_test = cache['raw_test']
    segment_feature = cache['segment_feature']
    
    # Получаем значения сегментного признака
    segment_values = raw_test[segment_feature].values
    
    # Создаём сегменты (квартили или бины)
    try:
        # Пробуем создать 5 бинов
        bins = np.percentile(segment_values, np.linspace(0, 100, 6))
        bins[0] = -np.inf
        bins[-1] = np.inf
        segments = np.digitize(segment_values, bins[1:-1])
    except:
        # Если не получается, используем уникальные значения
        segments = segment_values
    
    segment_table = build_segment_error_table(
        dataset_name=dataset_name,
        y_true=y_true,
        y_pred=y_pred,
        segment_values=segments,
        segment_feature=segment_feature,
        n_bins=5,
    )
    
    rows.append(segment_table)

error_by_segment = pd.concat(rows, ignore_index=True)

# Проверка колонок для error_by_segment
required_columns_task3 = {
    'dataset', 'segment_feature', 'segment', 'n', 'error_rate', 'false_positive_rate', 'false_negative_rate'
}
assert required_columns_task3.issubset(error_by_segment.columns)

# Сохранение CSV
threshold_tuning_results.to_csv(threshold_tuning_path, index=False)
cv_stability_results.to_csv(cv_stability_path, index=False)
error_by_segment.to_csv(error_by_segment_path, index=False)

print(f'Saved: {threshold_tuning_path}')
print(f'Saved: {cv_stability_path}')
print(f'Saved: {error_by_segment_path}')

display(error_by_segment)

Saved: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\threshold_tuning_results.csv
Saved: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\cv_stability_results.csv
Saved: C:\Users\melni\Desktop\ВУЗ\3курс\MathBigDataandmachinemodels\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\error_by_segment.csv


,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
0,medical,age,"(-0.001, 1.0]",72,0.222222,0.138889,0.083333
1,medical,age,"(1.6, 3.0]",70,0.385714,0.300000,0.085714
2,medical,age,"(3.0, 4.0]",38,0.421053,0.394737,0.026316
3,finance,credit_score,"(-0.001, 4.0]",220,0.322727,0.168182,0.154545
